# Data Preprocessing
This notebook covers dataset cleaning, inspection, and text normalization steps for fake
news detection.

Before building any machine learning models, it is essential to preprocess the data.
This includes standard steps such as converting text to lowercase, removing extra spaces,
and eliminating special characters to ensure consistency.

However, an important first step is to **visually inspect the dataset** before applying any
cleaning procedures.
For example, duplicated rows should be carefully examined — while duplicates are often
removed,
they may artificially inflate model performance and lead to overfitting if not handled
properly.

Below is the preprocessing code used in this project.

In [ ]:
import pandas as pd
import re

# Load data
df_true = pd.read_csv('DataSet_Misinfo_TRUE.csv')
df_fake = pd.read_csv('DataSet_Misinfo_FAKE.csv')

# label true and fake articles
df_true["label"] = 0
df_fake["label"] = 1

# combine two datasets
df = pd.concat([df_true, df_fake], ignore_index=True)

print(df.head())
print(df.shape)
print(df.duplicated().sum())

# drop unnecessary column
df = df.drop(columns=["Unnamed: 0"])

# save the combined dataset
df.to_csv("combined_dataset.csv", index=False)

# Remove rows with missing values to ensure model consistency
# (Missing text can break preprocessing or embedding steps)
df.isna().sum()
df.dropna(how='any', inplace=True)  # remove every row that contains at least one Na
df.info()

# examine duplicated rows
print("Number of duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

# create a function called clean_text
def clean_text(text):
    if not isinstance(text, str):  # check if the text is a string
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)  # remove urls
    text = re.sub(r"\s+", " ", text)  # normalize whitespace \s+ means (spaces tabs newlines)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text) # remove special characters
    return text.strip()  # remove whitespace from beginning and end


# Apply cleaning to text column only
df["text"] = df["text"].apply(clean_text)
print(df.head())


## Tokenization

Tokenization is a fundamental step in Natural Language Processing (NLP) that involves breaking down raw text into smaller, meaningful units called *tokens* (e.g., words or subwords).

This process helps transform unstructured text into a structured format that can be further processed by machine learning models. While tokenization itself does not directly convert text into numerical values, it prepares the text for subsequent steps such as vectorization (e.g., TF-IDF or embeddings), where tokens are mapped to numerical representations.

By standardizing and structuring the text, tokenization plays a key role in reducing complexity and enabling models to learn patterns for tasks such as classification and clustering.



## TF-IDF Representation

In this project, I used two text representation methods. The first is **TF-IDF (Term Frequency–Inverse Document Frequency)**.

TF-IDF converts text into numerical features based on how frequently words appear in a document relative to their frequency across the entire dataset. Words that appear often in a specific article but are rare across all articles receive higher weights, making them more informative for distinguishing documents.

However, TF-IDF assumes that words are independent of each other and does not capture word order or contextual meaning. As a result, the representation is based purely on word frequency rather than semantic relationships.

TF-IDF is useful when the goal is to capture word importance, **writing style, and surface-level textual patterns**. However, it does not preserve semantic context, which is why it is often less suitable than embeddings for meaning-based tasks.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Step 1: Create TF-IDF Matrix
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2), # include unigrams and bigrams
    max_df=0.95,        # removes words that appear in more than 95% documents
    min_df=5            # removes words that appear in fewer than 5 documents
)

X_tfidf = vectorizer.fit_transform(df["text"])
print(X_tfidf.shape)

# Step 2: Dimensionality Reduction (SVD)
# Truncated SVD reduces the high-dimensional TF-IDF matrix into a lower-dimensional latent space that preserves the most important document-level structure.
svd = TruncatedSVD(n_components=200, random_state=42)  # n_components will be changed according to the clusters
X_reduced = svd.fit_transform(X_tfidf)

# Step 3: Inspect Important Words in a Document
feature_names = vectorizer.get_feature_names_out()

doc_index = 0 # select one docment
row = X_tfidf[doc_index].toarray()[0]

# Get indices of top 10 TF-IDF values
top_indices = np.argsort(row)[-10:]

top_words = feature_names[top_indices]
top_values = row[top_indices]

print("\nTop TF-IDF words in document:")
for word, value in zip(top_words, top_values):
    print(f"{word}: {value:.4f}")


### Limitation of TF-IDF

While TF-IDF is effective for capturing word importance, it does not preserve semantic meaning or relationships between words. This limitation motivates the use of embedding-based methods (e.g., GPT embeddings), which capture deeper contextual information.

## ChatGPT API Embeddings

The second approach used in this project is **GPT-based text embeddings via the ChatGPT API**.

This method leverages a **pre-trained deep learning model** to convert raw text into dense numerical vectors that capture **semantic meaning**. Unlike TF-IDF, which relies on word frequency, GPT embeddings encode contextual relationships between words, allowing similar texts to be represented closer together in the embedding space.

As a result, this approach preserves the **semantic structure of the text**, making it particularly effective for tasks such as clustering and similarity analysis. In this project, these embeddings enable the use of unsupervised learning methods to explore the latent structure of fake news data.

Additionally, using a pre-trained API allows access to powerful models without the need for training from scratch, making it both efficient and cost-effectiv.

### Practical Considerations for GPT Embeddings

When using GPT-based embeddings through an API, there are several practical limitations to consider.

First, there are limits on the maximum input size (measured in tokens), meaning that very long text must be truncated or split into smaller chunks before processing.

Second, the API enforces rate limits, which restrict how many requests can be made within a certain time period. If too many requests are sent too quickly, the API may return errors.

Therefore, it is important to:
- monitor input length (token count)
- batch or chunk long texts when necessary
- implement delays or retries to handle rate limits

These considerations are important for efficiently processing large datasets when using GPT embeddings.

In [ ]:
from openai import OpenAI
from openai import RateLimitError
import time

# Initialize OpenAI client
client = OpenAI()
print(client)

# Test embedding model
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="This is a test sentence."
)

print(len(response.data[0].embedding))  #1536 dimensions vs. 200 dimensions


# Step 1: Stratified Sampling
df_sample, _ = train_test_split(
    df,
    train_size=6000,
    stratify=df["label"],
    random_state=42
)

texts_sample = df_sample["text"].astype(
    str).tolist()  # converts the pandas series into a python list because openai does not accept a ps

# Step 2: Embedding Function
# Create a function to get the embedding vlaues
def embed_texts(
        texts,
        model="text-embedding-3-small",
        batch_size=100):  # 100 texts per API request
    embs = []  # storage
    MAX_CHARS = 6000  # prevent sending extremely long articles to stabilizes API

    # loop through texts in batches
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        # Clean batch
        clean_batch = []
        #remove non-string entries; remove empty strings; strips whitespace; truncates long doc to 60000 characters
        for t in batch:
            if not isinstance(t, str):
                continue
            t = t.strip()
            if len(t) == 0:
                continue
            clean_batch.append(t[:MAX_CHARS])
        # Skip if batch becomes empty
        if len(clean_batch) == 0:
            continue
        # call the openai embedding API
        while True:
            try:
                resp = client.embeddings.create(model=model, input=clean_batch)
                # extract embedding vectors
                embs.extend([d.embedding for d in resp.data])
                time.sleep(1)
                break
            #when hit the token/min limit wait and retry
            except RateLimitError:
                print("Rate limit hit - waiting 5 seconds...")
                time.sleep(5)
        # progress print every 10 batch when batch_size = 200 it prints every 2000 doc
        if i % (batch_size * 10) == 0:
            print(f"Embedded {i} / {len(texts)}")
    # return final matrix converts the list of lists into a np array
    # each row is one article and each column is one embedding dimension
    return np.array(embs, dtype=np.float32)

# Step 3: Generate Embeddings
X_embed = embed_texts(texts_sample, batch_size=100)
print(X_embed.shape)
# (6000, 1536)


# Step 4: Save Results
# IMPORTANT: Avoid recomputing (costly API usage)
np.save("X_embed_6000.npy", X_embed)
df_sample.reset_index(drop=True).to_csv("df_sample_6000.csv", index=False)

# Step 5: Normalize Embeddings
from sklearn.preprocessing import normalize

# Normalize vectors
X_embed_norm = normalize(X_embed)  # ignores magnitude and focuses on direction (disregard the length of each article)
